# plot_04. Slide figures

Figures for talks, at one grid point and season of one model: the forcings revealed one at a time (a frame per step, all with the same axes so they can be flipped through), and the historical and hist-nat ensembles as members and as quantile plumes. Then, at the end, how the forced-response figures of notebooks plot_03 and plot_05 are made: how a count of models comes from one test per model, what the 3 × 3 colours of the joint maps mean, and the forcings' changes revealed one at a time. Everything is saved under `figures/<variable>/slides/` in this repository.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
from pathlib import Path
import warnings
from importlib import reload

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from plotting_modules import formatting
from extant import config, preprocessing as pp
from extant.config import ALPHA, FORCED_EXPERIMENTS, WINDOW
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

from functools import partial

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from plotting_modules import utils
from plotting_modules.constants import FORCING_COLORS, FORCING_REVEAL_ORDER, SEASONS
from extant import datatree, quantiles as qc
from extant.plots.slides import label_band_edges, label_line_end
from extant import significance as sig
from extant.plots import antarctica, charts, era5_evaluation as evp, forced_response as fr, slides

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook; the slides are drawn
#(t): in fr.SLIDES's
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [2]:
VARIABLE = config.VARIABLES['tas']

SLIDE_MODEL = config.PLOT_SEL['model']
SLIDE_SEASON = 'DJF'
SLIDE_POINT = config.POINT
POINT_LABEL = formatting.format_latlon(SLIDE_POINT)

#(t): For the forced-response figures (the end of this notebook): the experiment shown, and the fraction of
#(t): models that makes a change robust
SLIDE_EXPERIMENT = config.PLOT_SEL['experiment']
ROBUST = config.SIGNIFICANT_THRESHOLD
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): Where the slides go (this repository's figures folder), and where notebook calc_02's results are: JASMIN's scratch
SLIDE_DIR = Path('../figures') / VARIABLE.name / 'slides'
SLIDE_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SLIDE_DIR, SCRATCH_DIR

## Open the Data

Run **one** of the next two sections; both leave `lesfmip_season_tree`, and `RESULTS_DIR`, the folder of notebook calc_02's saved results the forced-response figures at the end open.

### Sample data

In [3]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

16:41:44 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/CanESM5_hist-GHG_sample.nc as /CanESM5/hist-GHG
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/CanESM5_hist-aer_sample.nc as /CanESM5/hist-aer
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/CanESM5_hist-nat_sample.nc as /CanESM5/hist-nat
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/CanESM5_hist-totalO3_sample.nc as /CanESM5/hist-totalO3
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/CanESM5_historical_sample.nc as /CanESM5/historical
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/HadGEM3-GC31-LL_hist-GHG_sample.nc as /HadGEM3-GC31-LL/hist-GHG
16:41:46 INFO    extant.loading | opening /home/users/aborow/ExtAnt-initial/tests/data/HadGEM3-GC31-LL_hist-aer_sample.nc as /HadGEM3-GC31-LL/hist-aer
16:41:46 IN

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]
sample_names

In [ ]:
#(t): Each file of config.MAIN_EXPERIMENTS opened, as a DataTree /<model>/<experiment>
lesfmip_sample_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                             for path, (model, experiment) in zip(sample_files, sample_names)
                                             if experiment in config.MAIN_EXPERIMENTS})
lesfmip_sample_tree

In [ ]:
#(t): Seasonal means in analysis units, through the same pre-processing as notebook calc_01
lesfmip_season_tree = pp.seasonal_tree(lesfmip_sample_tree, VARIABLE.name).compute()
lesfmip_season_tree

In [ ]:
#(t): Notebook calc_02's results from the sample
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

### Full data

In [4]:
#(t): The seasonal means notebook calc_01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
seasonal_path

reading /work/scratch-pw4/aborow/ExtAnt/seasonal/lesfmip_tas.zarr


/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be suppor

In [ ]:
#(t): Opened lazily
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
lesfmip_season_tree

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)
lesfmip_season_tree

In [ ]:
#(t): Notebook calc_02's results from the full data
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

### What was opened

Whichever section ran: how many members each model ran for each experiment. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=config.MAIN_EXPERIMENTS)
member_counts_df

In [ ]:
#(t): The members: each model (rows) and experiment (columns), grey where there are none
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.9 * member_counts_df.shape[1] + 0.5,
                                                  panel_height=0.4 * member_counts_df.shape[0] + 0.3, left_margin=1.8),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
counts_image = methods.plot_member_counts(member_counts_df, ax=axes[0, 0])
fig.colorbar(counts_image, ax=axes, location='bottom', shrink=0.6, label='Members')
fig.suptitle('LESFMIP members per model and experiment', x=0.01, ha='left');

## Grid Point

Every experiment of `SLIDE_MODEL` at `SLIDE_POINT`: a few MB, so loaded into memory.

In [5]:
#(t): Every experiment of SLIDE_MODEL at the point, all seasons
point_tree = lesfmip_season_tree[SLIDE_MODEL].sel(**SLIDE_POINT).load()
point_members = {forcing: point_tree[forcing][VARIABLE.name].sel(season=SLIDE_SEASON)
                 for forcing in FORCING_REVEAL_ORDER if forcing in point_tree.children}
point_members['historical']

<xarray.DataArray 'tas' (member: 65, year: 164)> Size: 43kB
-33.05 -33.0 -32.34 -31.74 -31.94 -31.44 ... -30.72 -32.04 -29.95 -30.04 -30.0
Coordinates:
    lon      float64 8B 120.0
    lat      float64 8B -82.5
    season   <U3 12B 'DJF'
  * member   (member) object 520B 'r10i1p1f1' 'r10i1p2f1' ... 'r9i1p2f1'
  * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2010 2011 2012 2013


## Build-up: One Forcing at a Time

First one member of each experiment, then the ensemble means on the same axes, then the means zoomed in. The dashed line is hist-nat's 1850–1900 mean.

In [6]:
single_member = {forcing: da.isel(member=0) for forcing, da in point_members.items()}
ensemble_mean = {forcing: da.mean('member') for forcing, da in point_members.items()}
baseline = float(ensemble_mean['hist-nat'].sel(year=slice(1850, 1900)).mean())

#(t): One y range for all the single-member and ensemble-mean frames, so they line up when flipped through
ylim = utils.shared_ylim([da.values for da in single_member.values()], include=[baseline])

In [ ]:
#(t): One member of each forcing, revealed in FORCING_REVEAL_ORDER: a frame per step (the first with no line yet), all
#(t): on the same axes (ylim), so they line up when flipped through
single_order = [forcing for forcing in FORCING_REVEAL_ORDER if forcing in single_member]
single_frames = []
with plt.rc_context(fr.SLIDES.rc):
    for step in range(len(single_order) + 1):
        fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, panel_width=8.0,
                                                          panel_height=3.6, top_margin=1.0), layout='constrained')
        axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
        slides.plot_buildup(single_member, single_order[:step], ylim, baseline=baseline, ax=axes[0, 0])
        axes[0, 0].set_xlabel('Year')
        axes[0, 0].set_ylabel(VARIABLE.label)
        slides.plot_buildup_legend(single_order, single_order[:step], fig)
        single_frames.append(utils.save_frame(fig, SLIDE_DIR / 'buildup_single_member', step,
                                              single_order[step - 1] if step else 'empty'))
[frame.name for frame in single_frames]

In [ ]:
#(t): The ensemble means on the same axes, historical straight after hist-nat, a frame per step; then all of them zoomed
#(t): in, none faded
mean_order = [f for f in ['hist-nat', 'historical', 'hist-GHG', 'hist-aer', 'hist-totalO3'] if f in ensemble_mean]
mean_dir = SLIDE_DIR / 'buildup_ensemble_mean'
zoom_ylim = utils.shared_ylim([da.values for da in ensemble_mean.values()], include=[baseline])
with plt.rc_context(fr.SLIDES.rc):
    for step in range(1, len(mean_order) + 2):
        #(c): The last frame is the zoom: every mean, on the means' own range
        zoom = step == len(mean_order) + 1
        fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, panel_width=8.0,
                                                          panel_height=3.6, top_margin=1.0), layout='constrained')
        axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
        slides.plot_buildup(ensemble_mean, mean_order[:step], zoom_ylim if zoom else ylim, fade_earlier=not zoom,
                            baseline=baseline, ax=axes[0, 0])
        axes[0, 0].set_xlabel('Year')
        axes[0, 0].set_ylabel(VARIABLE.label)
        slides.plot_buildup_legend(mean_order, mean_order[:step], fig)
        utils.save_frame(fig, mean_dir, step, 'zoom' if zoom else mean_order[step - 1], close=not zoom)

## historical Against hist-nat

Every member, then the same as quantile plumes: the rolling 21-year quantiles of all members pooled, in nested bands from the 10–90% range to the 40–60% range around the median.

In [9]:
PLUME_QUANTILES = np.round(np.arange(0.1, 1, 0.1), 2)
QUANTILE_PAIRS = np.array(list(zip(PLUME_QUANTILES[:len(PLUME_QUANTILES) // 2], PLUME_QUANTILES[::-1]))).round(2)

point_quantiles_da = datatree.reduce_to_dataset(
    point_tree,
    partial(qc.rolling_quantile, rolling_dim='year', window=config.WINDOW, quantiles=PLUME_QUANTILES,
            extra_dims=['member']),
    dims=['experiment'],
)[VARIABLE.name].compute()
historical_quantiles_da = point_quantiles_da.sel(experiment='historical', season=SLIDE_SEASON)
histnat_quantiles_da = point_quantiles_da.sel(experiment='hist-nat', season=SLIDE_SEASON)
label_x = historical_quantiles_da.year.values[-1]
QUANTILE_PAIRS

array([[0.1, 0.9],
       [0.2, 0.8],
       [0.3, 0.7],
       [0.4, 0.6]])

In [10]:
#(t): Quick look: historical's rolling quantiles (the plume's band edges), one line per quantile
historical_quantiles_da.plot.line(x='year', hue='quantile');

In [ ]:
#(t): Every member of historical and hist-nat
with plt.rc_context(fr.SLIDES.rc):
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, panel_width=10.5,
                                                      panel_height=4.8, left_margin=0.9), layout='constrained')
    axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
    for forcing, alpha in [('historical', 0.6), ('hist-nat', 0.3)]:
        charts.plot_lines(point_members[forcing], 'year', ax=axes[0, 0], color=FORCING_COLORS[forcing], alpha=alpha,
                          linewidth=0.8)
    axes[0, 0].legend(handles=[Patch(facecolor=FORCING_COLORS[f], label=f) for f in ('historical', 'hist-nat')],
                      loc='upper left', frameon=False)
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel(VARIABLE.label)
    axes[0, 0].set_title(f'{SLIDE_SEASON} ({POINT_LABEL})', loc='left')
fig.savefig(SLIDE_DIR / 'plume_lines.png', dpi=250, bbox_inches='tight')

In [ ]:
#(t): As quantile plumes, hist-nat's and historical's (nested bands, the median black), historical's band edges
#(t): labelled; the legend's patches as strong as the bands' middle, where they all overlap
band_alpha = 0.35
core_alpha = 1 - (1 - band_alpha) ** len(QUANTILE_PAIRS)
with plt.rc_context(fr.SLIDES.rc):
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, panel_width=10.0,
                                                      panel_height=4.8, left_margin=0.9), layout='constrained')
    axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
    for forcing, quantiles_da in [('hist-nat', histnat_quantiles_da), ('historical', historical_quantiles_da)]:
        charts.plot_quantile_bands(quantiles_da, QUANTILE_PAIRS, 'year', ax=axes[0, 0], color=FORCING_COLORS[forcing],
                                   alpha=band_alpha, edge_width=0.2)
        charts.plot_lines(quantiles_da.sel(quantile=0.5, method='nearest'), 'year', ax=axes[0, 0], color='k',
                          linewidth=1.8, zorder=3)
    axes[0, 0].legend(handles=[Patch(facecolor=FORCING_COLORS['historical'], alpha=core_alpha, label='historical'),
                               Patch(facecolor=FORCING_COLORS['hist-nat'], alpha=core_alpha, label='hist-nat'),
                               Line2D([], [], color='k', lw=2, label='Ensemble median')], loc='upper left',
                      frameon=False)
    label_band_edges(axes[0, 0], historical_quantiles_da, QUANTILE_PAIRS, label_x, color=FORCING_COLORS['historical'],
                     fontsize='small')
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel(VARIABLE.label)
    axes[0, 0].set_title(f'{SLIDE_SEASON} ({POINT_LABEL})', loc='left')
fig.savefig(SLIDE_DIR / 'plume_percentiles.png', dpi=250, bbox_inches='tight')

In [ ]:
#(t): The same, labelled with the experiment names instead of a legend
with plt.rc_context(fr.SLIDES.rc):
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, panel_width=10.0,
                                                      panel_height=4.8, left_margin=0.9), layout='constrained')
    axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
    for forcing, quantiles_da in [('hist-nat', histnat_quantiles_da), ('historical', historical_quantiles_da)]:
        charts.plot_quantile_bands(quantiles_da, QUANTILE_PAIRS, 'year', ax=axes[0, 0], color=FORCING_COLORS[forcing],
                                   alpha=band_alpha, edge_width=0.2)
        charts.plot_lines(quantiles_da.sel(quantile=0.5, method='nearest'), 'year', ax=axes[0, 0], color='k',
                          linewidth=1.8, zorder=3)
        label_line_end(axes[0, 0], label_x, quantiles_da.sel(quantile=0.5, method='nearest').isel(year=-1), forcing,
                       FORCING_COLORS[forcing], fontweight='bold')
    label_band_edges(axes[0, 0], historical_quantiles_da, QUANTILE_PAIRS, label_x, color=FORCING_COLORS['historical'],
                     fontsize='small')
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel(VARIABLE.label)
    axes[0, 0].set_title(SLIDE_SEASON, loc='left')
fig.savefig(SLIDE_DIR / 'plume_labelled.png', dpi=250, bbox_inches='tight')

## How the Forced-Response Figures Are Made

Three pictures for a talk about notebooks plot_03 and plot_05: where a count of models comes from, how to read the joint colours, and the forcings' changes revealed one at a time. They use notebook calc_02's saved results.

In [ ]:
#(t): The hist-nat bootstrap of the mean change (notebook calc_02, section 4)
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()
mean_bootstrap_ds

In [15]:
#(t): The width test (notebook calc_02, section 2)
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()
qrange_change_ds

16:41:54 INFO    extant.storage | opening /work/scratch-pw4/aborow/ExtAnt/results/tas/full/qrange_change.zarr


/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be suppor

### Counting the models

At `SLIDE_POINT` in `SLIDE_SEASON`: each model's hist-nat and `SLIDE_EXPERIMENT`, both over the final 21 years (members and years pooled), the hist-nat bootstrap of the change in their means, and its verdict. Counted, the verdicts give the number on the count map (notebook plot_03, section 1.4) at the starred point.

In [16]:
#(t): Every model at the point
all_models_point_tree = lesfmip_season_tree.sel(**SLIDE_POINT)

ValueError: cannot supply selection options {'method': 'nearest', 'tolerance': None} for dimension 'lon'that has no associated coordinate or index

In [ ]:
#(t): in SLIDE_SEASON
all_models_season_tree = all_models_point_tree.sel(season=SLIDE_SEASON)

In [ ]:
#(t): over the final WINDOW years, loaded into memory (a few kB)
point_final_tree = all_models_season_tree.isel(year=slice(-WINDOW, None)).load()

In [ ]:
#(t): hist-nat's values in each model over the same final years, what the change is measured from, on (member, year)
point_hist_nat = {node.parent.name: node[VARIABLE.name] for node in point_final_tree.match('*/hist-nat').leaves}
list(point_hist_nat)

In [ ]:
#(t): SLIDE_EXPERIMENT's
point_experiment = {node.parent.name: node[VARIABLE.name]
                    for node in point_final_tree.match(f'*/{SLIDE_EXPERIMENT}').leaves}
list(point_experiment)

In [ ]:
#(t): The test at the point, every model: the change in the mean and its p-value
point_test_ds = mean_bootstrap_ds.sel(experiment=SLIDE_EXPERIMENT, season=SLIDE_SEASON).sel(**SLIDE_POINT)
point_test_ds.to_dataframe()[['mean_change', 'pvalue']]

In [ ]:
#(t): The test everywhere, SLIDE_EXPERIMENT in SLIDE_SEASON
season_test_ds = mean_bootstrap_ds.sel(experiment=SLIDE_EXPERIMENT, season=SLIDE_SEASON)

In [ ]:
#(t): How many models have a significant increase, and a decrease, at every point
slide_counts_da = sig.significant_counts(season_test_ds['mean_change'], season_test_ds['pvalue'] < ALPHA)

In [ ]:
#(t): The verdicts counted at the point: how many models have a significant increase, and a decrease, which is the
#(t): number the count map gives there
point_significant_da = point_test_ds['pvalue'] < ALPHA
n_up = int(((point_test_ds['mean_change'] > 0) & point_significant_da).sum())
n_down = int(((point_test_ds['mean_change'] < 0) & point_significant_da).sum())
n_up, n_down

In [ ]:
#(t): The count map as one number per point: increases count up, decreases down (as notebook plot_03's count maps)
signed_counts_da = slide_counts_da.sel(direction='increase') - slide_counts_da.sel(direction='decrease')
signed_counts_da

In [ ]:
#(t): The models, and how the tests are laid out: two rows if there are more than two
models = list(point_hist_nat)
n_test_cols = int(np.ceil(len(models) / 2)) if len(models) > 2 else len(models)
n_test_rows = int(np.ceil(len(models) / n_test_cols))

In [ ]:
#(t): From one test per model to a count: each model's test at the point (left), the verdicts counted (middle), and the
#(t): count map of SLIDE_EXPERIMENT with the point starred (right)
map_width, map_height = antarctica.map_figure_size(n_rows=1, n_cols=1, n_colorbar_rows=1, panel_size=3.6)
with plt.rc_context(fr.SLIDES.rc):
    fig = plt.figure(figsize=(3.0 * n_test_cols + 1.4 + map_width, max(2.6 * n_test_rows + 1.2, map_height)))
    tests_fig, arrow_fig, map_fig = fig.subfigures(1, 3, width_ratios=[3.0 * n_test_cols, 1.4, map_width])

    #(c): Left: each model's test, on one x axis. The map beside them means the figure isn't constrained, so the
    #(c): room round the panels is in inches (charts.margins): above for the titles, below for the legend
    test_grid = tests_fig.add_gridspec(n_test_rows, n_test_cols, wspace=0.15, hspace=0.6,
                                       **charts.margins(tests_fig, left=0.3, bottom=0.9))
    test_axes = test_grid.subplots(sharex=True, squeeze=False)
    for ax, model in zip(test_axes.flat, models):
        slides.plot_point_test(point_hist_nat[model], point_experiment[model],
                               float(point_test_ds['mean_change'].sel(model=model)),
                               float(point_test_ds['pvalue'].sel(model=model)), SLIDE_EXPERIMENT, ax=ax, alpha=ALPHA,
                               units=VARIABLE.units)
        ax.set_title(model, loc='left', fontsize=11)
    for ax in test_axes.flat[len(models):]:
        ax.axis('off')
    #(c): The axes are shared: a few ticks, set once, so their labels fit
    test_axes[0, 0].xaxis.set_major_locator(plt.MaxNLocator(4))
    tests_fig.legend(*test_axes[0, 0].get_legend_handles_labels(), loc='lower center', ncols=2, frameon=False)

    #(c): Middle: an arrow, and the verdicts counted
    arrow_ax = arrow_fig.add_subplot()
    arrow_ax.axis('off')
    arrow_ax.annotate('', xy=(0.95, 0.5), xytext=(0.05, 0.5), xycoords='axes fraction',
                      arrowprops=dict(arrowstyle='-|>', lw=2.5, color='0.3', mutation_scale=25))
    arrow_ax.text(0.5, 0.58, 'count the\nverdicts', ha='center', va='bottom', fontsize=11, color='0.3')
    arrow_ax.text(0.5, 0.42, f'{n_up} up\n{n_down} down\nof {len(models)}', ha='center', va='top', fontsize=12,
                  fontweight='bold')

    #(c): Right: the count map, with the point starred
    map_axes = antarctica.map_axes(map_fig, n_rows=1, n_cols=1)
    map_cax = antarctica.colorbar_axis_below(map_axes)
    antarctica.heatmaps_of_antarctica(signed_counts_da, None, None, axes=map_axes, cax=map_cax,
                                      **fr.count_scale(len(models), 'decrease (blue) or increase (red)').colours())
    map_axes[0, 0].plot(float(point_test_ds['lon']), float(point_test_ds['lat']), marker='*', ms=18, color='gold',
                        mec='k', mew=1.2, transform=ccrs.PlateCarree(), zorder=5)
    antarctica.label_columns(map_axes, [f'{SLIDE_EXPERIMENT}: models with a\nsignificant change'], fontsize=11)
    fig.suptitle(f'From one test per model to a count: {SLIDE_EXPERIMENT}, {SLIDE_SEASON} at {POINT_LABEL}',
                 fontsize=14, fontweight='bold', x=0.04, ha='left')

In [ ]:
fig.savefig(SLIDE_DIR / 'count_schematic.png', dpi=250, bbox_inches='tight')

### Reading the joint colours

The 3 × 3 key of the joint maps (notebook plot_03 section 5, notebook plot_05 Fig. 4), each colour drawn as what it means for the distribution: shifted down, not or up, and narrower, the same or wider.

In [ ]:
#(t): The 3 × 3 key of the joint maps drawn as what each colour means: hist-nat's distribution (grey, dashed) and a
#(t): changed one, shifted down, not or up (rows, bottom to top) and narrower, the same or wider (columns)
joint_rows, joint_columns, joint_row_title, joint_column_title = scales.joint.key
with plt.rc_context(fr.SLIDES.rc):
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=3, n_cols=3, n_legends=1, panel_width=2.2,
                                                      panel_height=1.8, left_margin=2.0, column_gap=0.1,
                                                      row_gap=0.1), layout='constrained')
    axes = fig.add_gridspec(3, 3).subplots(squeeze=False)
    for i, shift in enumerate((-1.2, 0.0, 1.2)):
        for j, width in enumerate((0.65, 1.0, 1.5)):
            #(c): Class i * 3 + j is row i of the key from the bottom; the middle class (no change) is not filled, which
            #(c): would hide hist-nat's
            joint_class = i * len(joint_columns) + j
            slides.plot_joint_key_cell(shift, width, scales.joint.cmap(joint_class), fill=joint_class != 4,
                                       ax=axes[len(joint_rows) - 1 - i, j])
    for ax, column in zip(axes[-1], joint_columns):
        ax.set_xlabel(column)
    for ax, row in zip(axes[::-1, 0], joint_rows):
        ax.set_ylabel(row, rotation=0, ha='right', va='center')
    axes[0, 1].set_title(f'{joint_column_title} of the distribution', loc='center', color=evp.INK_2)
    axes[1, 0].annotate(joint_row_title, (-0.6, 0.5), xycoords='axes fraction', rotation=90, ha='center', va='center',
                        color=evp.INK_2)
    fig.legend(handles=[Line2D([], [], color='0.35', lw=1.4, ls=(0, (3, 2)), label='hist-nat'),
                        Line2D([], [], color='k', lw=1.4, label='the experiment')], loc='outside lower center', ncols=2,
               frameon=False)
    fig.suptitle('Reading the colours: how the distribution changed', x=0.01, ha='left')

In [ ]:
fig.savefig(SLIDE_DIR / 'joint_key.png', dpi=250, bbox_inches='tight')

### Forcings one at a time

The multi-model median change in the mean and in the width, every season, with one forcing added per frame (the order of `FORCED_EXPERIMENTS`), dotted where fewer than two-thirds of the models have a significant change of one sign. Every frame has the same canvas, so they can be flipped through.

In [ ]:
#(t): The multi-model median change in the mean
median_mean_change_da = mean_bootstrap_ds['mean_change'].median('model')
median_mean_change_da

In [ ]:
#(t): Where the models agree on a significant change in the mean
mean_robust_sign_da = sig.robust_sign(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'] < ALPHA, threshold=ROBUST)

In [ ]:
#(t): Not robust: the dots
mean_not_robust_da = mean_robust_sign_da == 0
mean_not_robust_da

In [ ]:
#(t): The frames: the mean, one experiment (row) more in each. Every frame is the same canvas, the rows still to
#(t): come empty maps and the colour bar and legend there from the start, so flipping through them only adds maps
mean_change_da = median_mean_change_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
mean_dots_da = mean_not_robust_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                               label='Not robust (fewer than 2/3 of the models significant)')
mean_frames = []
with plt.rc_context(fr.SLIDES.rc):
    for step in range(1, len(FORCED_EXPERIMENTS) + 1):
        #(c): The change's colour bar under the maps, then the dots' legend
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1, panel_size=2.4),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)

        #(c): The first `step` experiments, and their dots; the rest are empty maps
        antarctica.heatmaps_of_antarctica(mean_change_da.isel(experiment=slice(0, step)), 'experiment', 'season',
                                          axes=axes[:step], cax=cax, **scales.change.colours())
        antarctica.not_significant_dots_grid(mean_dots_da.isel(experiment=slice(0, step)), 'experiment', 'season',
                                             axes=axes[:step])
        for ax in axes[step:].ravel():
            antarctica.style_antarctic_ax(ax)

        antarctica.label_columns(axes, SEASONS, fontsize=fr.SLIDES.label_size)
        antarctica.label_rows(axes, FORCED_EXPERIMENTS, fontsize=fr.SLIDES.label_size)
        fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False)
        fig.suptitle('Change in the mean, final 21 years against hist-nat (multi-model median)', fontsize=fr.SLIDES.title_size)
        mean_frames.append(utils.save_frame(fig, SLIDE_DIR / 'reveal_mean_change', step, FORCED_EXPERIMENTS[step - 1]))
[frame.name for frame in mean_frames]

In [ ]:
#(t): The multi-model median change in the width
median_width_change_da = qrange_change_ds['qrange_change'].median('model')
median_width_change_da

In [ ]:
#(t): Where the models agree on a significant change in the width
width_robust_sign_da = sig.robust_sign(qrange_change_ds['qrange_change'], qrange_change_ds['qrange_significant'],
                                       threshold=ROBUST)

In [ ]:
#(t): Not robust: the dots
width_not_robust_da = width_robust_sign_da == 0
width_not_robust_da

In [ ]:
#(t): The frames: the width, one experiment (row) more in each. Every frame is the same canvas, the rows still to
#(t): come empty maps and the colour bar and legend there from the start, so flipping through them only adds maps
width_change_da = median_width_change_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
width_dots_da = width_not_robust_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                               label='Not robust (fewer than 2/3 of the models significant)')
width_frames = []
with plt.rc_context(fr.SLIDES.rc):
    for step in range(1, len(FORCED_EXPERIMENTS) + 1):
        #(c): The change's colour bar under the maps, then the dots' legend
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1, panel_size=2.4),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)

        #(c): The first `step` experiments, and their dots; the rest are empty maps
        antarctica.heatmaps_of_antarctica(width_change_da.isel(experiment=slice(0, step)), 'experiment', 'season',
                                          axes=axes[:step], cax=cax, **scales.spread_change.colours())
        antarctica.not_significant_dots_grid(width_dots_da.isel(experiment=slice(0, step)), 'experiment', 'season',
                                             axes=axes[:step])
        for ax in axes[step:].ravel():
            antarctica.style_antarctic_ax(ax)

        antarctica.label_columns(axes, SEASONS, fontsize=fr.SLIDES.label_size)
        antarctica.label_rows(axes, FORCED_EXPERIMENTS, fontsize=fr.SLIDES.label_size)
        fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False)
        fig.suptitle('Change in the width Q95 − Q05, final 21 years against hist-nat (multi-model median)', fontsize=fr.SLIDES.title_size)
        width_frames.append(utils.save_frame(fig, SLIDE_DIR / 'reveal_width_change', step,
                                             FORCED_EXPERIMENTS[step - 1]))
[frame.name for frame in width_frames]